# Week 7 — Quality Filtering of the NonLinLoc Catalogue
Parses all 2982 NLLoc .hyp hypocentre files, extracts the quality metrics
needed for the proposal's stated filters (max horizontal error <5 km,
azimuthal gap <180 deg, minimum 8 phases, mean RMS residual <0.5 s), and
produces the quality-filtered absolute location catalogue.

Field positions within each NLLoc line are confirmed against a real,
tested open-source parser (NonLinLocPy) cross-checked against NonLinLoc's
own documented example file - not guessed.

Horizontal error is approximated as sqrt(CovXX + CovYY) from the
STATISTICS line's covariance matrix - the standard practical proxy for
horizontal uncertainty, rather than the full confidence-ellipsoid
semi-major-axis calculation. Documented simplification, noted for the
methods section.

In [ ]:
import os, glob
import pandas as pd

PROJECT_DIR = os.path.expanduser("~/tvz_project")
LOC_DIR = os.path.join(PROJECT_DIR, "nonlinloc", "loc")

hyp_files = sorted(glob.glob(os.path.join(LOC_DIR, "*.grid0.loc.hyp")))
print(f"{len(hyp_files)} .hyp files found")

In [ ]:
def parse_hyp(path):
    result = {"file": os.path.basename(path)}
    with open(path) as f:
        for line in f:
            parts = line.split()
            if not parts:
                continue
            if parts[0] == "GEOGRAPHIC":
                # GEOGRAPHIC OT year month day hour min sec Lat lat Long lon Depth depth
                result["origin_time"] = f"{parts[2]}-{parts[3]}-{parts[4]}T{parts[5]}:{parts[6]}:{parts[7]}"
                result["lat"] = float(parts[9])
                result["lon"] = float(parts[11])
                result["depth_km"] = float(parts[13])
            elif parts[0] == "QUALITY":
                result["rms"] = float(parts[8])
                result["nphs"] = float(parts[10])
                result["gap"] = float(parts[12])
            elif parts[0] == "STATISTICS":
                cov_xx = float(parts[8])
                cov_yy = float(parts[14])
                result["horiz_error_km"] = (cov_xx + cov_yy) ** 0.5
    return result

records = [parse_hyp(f) for f in hyp_files]
catalog = pd.DataFrame(records)
print(f"Parsed {len(catalog)} events")
print(catalog.head())
print(f"\nMissing horiz_error_km (no STATISTICS line?): {catalog['horiz_error_km'].isna().sum()}")

In [ ]:
# Apply the proposal's stated quality filters
print("=== Before filtering ===")
print(f"Total events: {len(catalog)}")
print(catalog[["rms", "nphs", "gap", "horiz_error_km"]].describe())

filtered = catalog[
    (catalog["horiz_error_km"] < 5.0) &
    (catalog["gap"] < 180.0) &
    (catalog["nphs"] >= 8) &
    (catalog["rms"] < 0.5)
].copy()

print(f"\n=== After filtering ===")
print(f"Events retained: {len(filtered)} of {len(catalog)} ({100 * len(filtered) / len(catalog):.1f}%)")

# Break down rejection reasons for transparency
print("\nRejection breakdown (events failing each individual criterion):")
print(f"  horiz_error >= 5km: {(catalog['horiz_error_km'] >= 5.0).sum()}")
print(f"  gap >= 180deg:      {(catalog['gap'] >= 180.0).sum()}")
print(f"  nphs < 8:           {(catalog['nphs'] < 8).sum()}")
print(f"  rms >= 0.5s:        {(catalog['rms'] >= 0.5).sum()}")

In [ ]:
out_csv = os.path.join(PROJECT_DIR, "catalogs", "eqtransformer_nlloc_quality_filtered.csv")
raw_csv = os.path.join(PROJECT_DIR, "catalogs", "eqtransformer_nlloc_raw.csv")

catalog.to_csv(raw_csv, index=False)
filtered.to_csv(out_csv, index=False)
print(f"Saved raw catalog: {raw_csv}")
print(f"Saved quality-filtered catalog: {out_csv}")